# Bayes Series | Chapter 1: Naive Bayes Classifiers

> **Chapter Goal:** Understand how Bayes' theorem applied to classification gives us a family of fast, interpretable classifiers — and why they work despite making an assumption that is almost always wrong. Master Gaussian Naive Bayes (continuous features), Multinomial Naive Bayes (text classification), and Bernoulli Naive Bayes. Know exactly when to use each, when they fail, and how Laplace smoothing is a Bayesian MAP estimate.

---

## 1. Generative vs Discriminative Classifiers

There are two fundamentally different approaches to classification:

| | Generative | Discriminative |
| :--- | :--- | :--- |
| **Models** | $P(X, Y) = P(X|Y)P(Y)$ — the joint distribution | $P(Y|X)$ — directly models the decision boundary |
| **Classifier** | Apply Bayes' theorem to get $P(Y|X)$ | Directly predict $P(Y|X)$ |
| **Examples** | Naive Bayes, GDA, HMMs | Logistic Regression, SVM, Neural Networks |
| **Advantage** | Can generate new data; works with less data (strong assumptions help) | More flexible; doesn't need to model $P(X|Y)$ |
| **Disadvantage** | Assumptions about $P(X|Y)$ must hold; otherwise, discriminative wins | No insight into data generation; can't generate samples |

**Naive Bayes is generative.** It models how data was generated: first the class $Y$ is chosen (with probability $P(Y)$), then the features $X$ are generated given the class (with probability $P(X|Y)$).

---

## 2. The Bayes Optimal Classifier

Given features $\vec{x}$, predict the class that maximizes the posterior probability:

$$\hat{y} = \arg\max_c P(Y=c | \vec{x})$$

Apply Bayes' theorem:
$$P(Y=c | \vec{x}) = \frac{P(\vec{x} | Y=c) \cdot P(Y=c)}{P(\vec{x})}$$

Since $P(\vec{x})$ (the evidence) is the same for all classes, we can drop it:
$$\hat{y} = \arg\max_c P(\vec{x} | Y=c) \cdot P(Y=c)$$

$$\underbrace{\hat{y}}_\text{prediction} = \arg\max_c \underbrace{P(\vec{x}|Y=c)}_\text{class-conditional likelihood} \cdot \underbrace{P(Y=c)}_\text{class prior}$$

We need two things:
1. **Class prior** $P(Y=c)$ — how frequent is class $c$? (Easy: fraction of training points in class $c$)
2. **Class-conditional likelihood** $P(\vec{x}|Y=c)$ — given class $c$, how probable is feature vector $\vec{x}$?

The second one is the hard part. For $d$ features, modeling the full joint $P(\vec{x}|Y=c)$ requires enormous amounts of data. Enter the **naive** assumption.

---

## 3. The Naive Independence Assumption

**Assumption:** Features are **conditionally independent given the class**:
$$P(\vec{x} | Y=c) = P(x_1, x_2, ..., x_d | Y=c) = \prod_{j=1}^d P(x_j | Y=c)$$

This is the "naive" in Naive Bayes. It says: once you know the class, features are independent of each other.

### **Why This Is Almost Always Wrong**
Consider spam detection with features "free" and "money". These words co-occur in spam emails — they are highly dependent given class = spam. The independence assumption is violated.

### **Why It Often Works Anyway**
Despite being wrong, Naive Bayes often performs well because:
1. **Classification only needs the argmax**, not accurate probabilities. Even if the independence assumption gives wrong probability estimates, it may still rank classes correctly.
2. **Less parameters to estimate** — $O(d \cdot C)$ parameters instead of exponential. This reduces overfitting, especially with small $N$.
3. **The decision surface's properties** — for log-linear models (Gaussian NB with equal variance), NB gives an equivalent decision boundary to logistic regression.

### **With The Naive Assumption**
$$\hat{y} = \arg\max_c P(Y=c) \prod_{j=1}^d P(x_j | Y=c)$$

In log space (more numerically stable):
$$\hat{y} = \arg\max_c \left[ \log P(Y=c) + \sum_{j=1}^d \log P(x_j | Y=c) \right]$$

---

## 4. Gaussian Naive Bayes (Continuous Features)

**When to use:** Features are continuous and (roughly) normally distributed within each class.

### **Model**
Assume each feature $x_j$ follows a Gaussian within each class:
$$P(x_j | Y=c) = \mathcal{N}(x_j; \mu_{jc}, \sigma_{jc}^2)$$

Parameters: $\mu_{jc}$ (mean of feature $j$ in class $c$) and $\sigma_{jc}^2$ (variance of feature $j$ in class $c$).

### **MLE Parameter Estimation**
From training data $\{(\vec{x}_i, y_i)\}$, learn:
$$\hat{\pi}_c = P(Y=c) = \frac{N_c}{N} \quad \text{(fraction of class-}c\text{ samples)}$$
$$\hat{\mu}_{jc} = \frac{1}{N_c}\sum_{i: y_i=c} x_{ij} \quad \text{(per-class, per-feature mean)}$$
$$\hat{\sigma}_{jc}^2 = \frac{1}{N_c}\sum_{i: y_i=c} (x_{ij} - \hat{\mu}_{jc})^2 \quad \text{(per-class, per-feature variance)}$$

### **Prediction**
$$\hat{y} = \arg\max_c \left[ \log\hat{\pi}_c + \sum_{j=1}^d \left( -\frac{1}{2}\log(2\pi\hat{\sigma}_{jc}^2) - \frac{(x_j - \hat{\mu}_{jc})^2}{2\hat{\sigma}_{jc}^2} \right) \right]$$

### **Decision Boundary**
- **Different $\sigma_{jc}^2$ per class (general case):** Quadratic decision boundary.
- **Equal variance across classes $\sigma_{jc}^2 = \sigma_j^2$:** The quadratic terms cancel and the boundary is linear (equivalent to LDA).

---

## 5. Multinomial Naive Bayes (Text Classification)

**When to use:** Features are word counts (or term frequencies). Classic for spam detection, sentiment analysis, topic classification.

### **The Bag-of-Words Model**
Represent each document as a count vector $\vec{x} = (n_1, n_2, ..., n_V)$ where $n_w$ = count of word $w$ in the document, $V$ = vocabulary size.

The document is modeled as $n = \sum_w n_w$ words drawn i.i.d. from a Multinomial distribution:
$$P(\vec{x} | Y=c) \propto \prod_{w=1}^V \theta_{wc}^{n_w}$$

where $\theta_{wc} = P(\text{word } w | \text{class } c)$ (probability of word $w$ occurring given class $c$).

### **MLE Parameter Estimation**
$$\hat\theta_{wc} = \frac{\text{count of word } w \text{ in all class-}c \text{ documents}}{\text{total words in all class-}c \text{ documents}} = \frac{\sum_{i: y_i=c} n_{wi}}{\sum_{i: y_i=c} \sum_{w'} n_{w'i}}$$

### **The Zero-Frequency Problem**
If word $w$ never appears in class-$c$ training documents: $\hat\theta_{wc}^{\text{MLE}} = 0$.

Then the entire log-probability for class $c$ becomes $-\infty$ if any new document contains word $w$ — even if that document strongly resembles class $c$ in all other words. **One unseen word zeroes out the entire class.**

### **Laplace Smoothing (Add-$\alpha$) — A Bayesian Solution**
$$\hat\theta_{wc}^{\text{smooth}} = \frac{(\sum_{i: y_i=c} n_{wi}) + \alpha}{(\sum_{i: y_i=c} \sum_{w'} n_{w'i}) + \alpha V}$$

where $\alpha$ is the smoothing parameter (usually $\alpha = 1$ — "Laplace smoothing" or "add-one smoothing").

**What Laplace smoothing really is (Bayesian derivation):**
- Place a **Dirichlet prior** $\boldsymbol{\theta}_c \sim \text{Dir}(\alpha, \alpha, ..., \alpha)$ on word probabilities for class $c$.
- With equal $\alpha$ for all words: symmetric Dirichlet — expresses uniform uncertainty.
- The MAP estimate (multiply prior × likelihood, maximize) gives exactly the smoothed estimate above.
- $\alpha = 1$: add-one smoothing. $\alpha < 1$: Jeffreys prior. $\alpha \to 0$: MLE (no smoothing).

---

## 6. Bernoulli Naive Bayes (Binary Features)

**When to use:** Features are binary — presence/absence of each word, pixel on/off, feature exists or not.

### **Difference from Multinomial NB**
| | Multinomial NB | Bernoulli NB |
| :--- | :--- | :--- |
| Feature type | Word counts $n_w \in \{0,1,2,...\}$ | Word presence $x_w \in \{0, 1\}$ |
| Models | Frequency of words | Presence/absence of words |
| Ignores | — | Word counts (just presence) |
| Explicit non-occurrence | Only counts positive occurrences | Explicitly penalizes absent features |

### **Likelihood**
$$P(\vec{x}|Y=c) = \prod_{j=1}^d \theta_{jc}^{x_j} (1-\theta_{jc})^{1-x_j}$$

### **Key Difference in Prediction**
Bernoulli NB **penalizes the absence of features that are typical of the class.** If a word typically appears in class $c$ but is absent from the document, that's evidence against class $c$. Multinomial NB doesn't have this: a zero count just contributes nothing.

This makes Bernoulli NB better when **what's absent matters as much as what's present** (e.g., short documents where specific keywords signal the class).

---

## 7. Strengths, Weaknesses & When to Use

### **Strengths**
- ✅ **Extremely fast:** Training = one pass to count statistics. Prediction = a few multiplications.
- ✅ **Works with very small training data** — $O(d \cdot C)$ parameters vs $O(d^2)$ for a Gaussian model.
- ✅ **Handles irrelevant features reasonably** — independence assumption dilutes irrelevant feature effects.
- ✅ **No missing data problem** — can train and predict with missing features by marginalizing.
- ✅ **Interpretable** — $\theta_{jc}$ directly tells you how indicative each feature is for each class.
- ✅ **Online/incremental learning** — just accumulate sufficient statistics as new data arrives.

### **Weaknesses**
- ❌ **Independence assumption is almost always wrong** — correlated features are double-counted.
- ❌ **Poor probability calibration** — predicted class probabilities are often very extreme (near 0 or 1), not reliable confidence scores even when classification is correct.
- ❌ **Gaussian NB assumes Gaussian features** — badly calibrated if features are skewed, multimodal, or bounded.
- ❌ **Discrete NB can't handle continuous features naturally** (must discretize).

### **When to Use Naive Bayes**
| Use Case | Why |
| :--- | :--- |
| Text classification (spam, sentiment) | Works extremely well; fast; Multinomial NB is the default baseline |
| Very small datasets | Strong independence assumption reduces variance |
| Real-time prediction needed | Training and prediction are $O(Nd)$ — fastest among classifiers |
| Many classes | Scales linearly with $C$ — unlike most other classifiers |
| Strong baseline | Always try NB first; if it doesn't work, your problem may be complex |

---

## 8. Interview Q&A

**Q1: Why is Naive Bayes called "naive"?**
> Because it makes the naive assumption that all features are conditionally independent given the class label: $P(x_1,...,x_d|Y=c) = \prod_j P(x_j|Y=c)$. This is rarely true in practice but dramatically simplifies computation and parameter estimation.

**Q2: Derive the Naive Bayes classifier from Bayes' theorem.**
> Start with $\hat{y} = \arg\max_c P(Y=c|\vec{x})$. Apply Bayes: $P(Y=c|\vec{x}) \propto P(\vec{x}|Y=c)P(Y=c)$. Apply the naive independence assumption: $P(\vec{x}|Y=c) = \prod_j P(x_j|Y=c)$. Result: $\hat{y} = \arg\max_c \left[\log P(Y=c) + \sum_j \log P(x_j|Y=c)\right]$.

**Q3: Why does Naive Bayes often work despite the independence assumption being wrong?**
> Because classification only needs the correct argmax, not accurate marginal probabilities. Even if the independence assumption distorts the probability estimates, the class with the highest score (posterior) is often still the correct class. Furthermore, fewer parameters means less overfitting, which can offset the model misspecification.

**Q4: What is Laplace smoothing and why is it needed?**
> If a word appears in test data but not in any training document of class $c$, the MLE gives $\theta_{wc} = 0$, making $P(\text{doc}|Y=c) = 0$ regardless of all other evidence. Laplace smoothing adds a pseudo-count $\alpha$ to each word, giving $\hat\theta_{wc} = (n_{wc}+\alpha)/(N_c\cdot \bar{n} + \alpha V)$. This is the MAP estimate with a symmetric Dirichlet prior on word probabilities.

**Q5: When would you prefer Multinomial vs Bernoulli Naive Bayes?**
> Multinomial NB uses word counts and works well for longer documents where term frequency matters. Bernoulli NB uses binary presence/absence and also explicitly penalizes absent features — better for short documents or when what's absent is informative. In practice, try both and compare on a validation set.

**Q6: Can you use Naive Bayes for regression?**
> Not directly — it's a classifier. However, you could discretize the target into bins and classify. Or use Gaussian Process regression (also Bayesian) for regression. Naive Bayes is specifically designed for multi-class classification.

**Q7: How does Naive Bayes handle missing features at test time?**
> Simply omit the missing feature's likelihood term from the product. If $x_j$ is missing, compute $P(d|Y=c) = \prod_{j \text{ observed}} P(x_j|Y=c)$. This is a natural advantage of the generative model — you marginalize over the missing feature.

**Q8: Is Naive Bayes invariant to irrelevant features?**
> Partially. If a feature $x_j$ is completely independent of $Y$ (truly irrelevant), then $P(x_j|Y=c) \approx P(x_j|Y=c')$ for all classes. The log-likelihood contribution of that feature is approximately the same for all classes, so it cancels in the argmax. In practice, irrelevant features add noise but don't catastrophically break NB.

**Q9: What is the relationship between Naive Bayes and Logistic Regression?**
> For Gaussian NB with equal class variances: the log-posterior ratio is linear in the features — the same form as logistic regression. This means Gaussian NB (equal variance) and logistic regression have the same model family but are fit differently. NB finds the best generative model (assuming independence) while LR finds the best discriminative boundary. LR often wins with more data; NB can win with less data.

---

In [ ]:
# ─── CELL 1: Gaussian Naive Bayes From Scratch ────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, load_wine
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

class GaussianNaiveBayes:
    """Gaussian Naive Bayes from scratch."""
    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.n_classes_ = len(self.classes_)
        N, d = X.shape
        # Class priors
        self.log_priors_ = {}
        # Per-class per-feature Gaussian parameters
        self.mu_ = {}   # shape: (C, d)
        self.var_ = {}  # shape: (C, d)

        for c in self.classes_:
            Xc = X[y == c]
            self.log_priors_[c] = np.log(len(Xc) / N)
            self.mu_[c] = Xc.mean(axis=0)            # MLE mean per feature per class
            self.var_[c] = Xc.var(axis=0) + 1e-9     # MLE variance + smoothing
        return self

    def _log_likelihood(self, X, c):
        """Log P(X | Y=c) = sum_j log N(x_j; mu_jc, var_jc)"""
        mu, var = self.mu_[c], self.var_[c]
        # log N(x; mu, var) = -0.5*log(2π*var) - (x-mu)^2/(2*var)
        return np.sum(-0.5 * np.log(2 * np.pi * var)
                      - 0.5 * (X - mu)**2 / var, axis=1)  # shape: (N,)

    def predict_log_proba(self, X):
        """Log posterior: log P(Y=c | X) ∝ log P(X|Y=c) + log P(Y=c)"""
        log_posts = np.column_stack([
            self._log_likelihood(X, c) + self.log_priors_[c]
            for c in self.classes_
        ])  # shape: (N, C)
        return log_posts

    def predict(self, X):
        return self.classes_[self.predict_log_proba(X).argmax(axis=1)]

    def predict_proba(self, X):
        log_posts = self.predict_log_proba(X)
        # Normalize (log-sum-exp trick for numerical stability)
        log_posts -= log_posts.max(axis=1, keepdims=True)
        posts = np.exp(log_posts)
        return posts / posts.sum(axis=1, keepdims=True)


# Test on Iris and Wine
print("=" * 50)
for dataset_name, dataset in [('Iris', load_iris()), ('Wine', load_wine())]:
    X, y = dataset.data, dataset.target
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    
    # Our implementation
    gnb = GaussianNaiveBayes().fit(X_train, y_train)
    acc_mine = accuracy_score(y_test, gnb.predict(X_test))
    
    # sklearn's GNB
    gnb_sk = GaussianNB().fit(X_train, y_train)
    acc_sk = accuracy_score(y_test, gnb_sk.predict(X_test))
    
    print(f"{dataset_name}: My GNB={acc_mine:.4f}  Sklearn GNB={acc_sk:.4f}  Match={np.isclose(acc_mine, acc_sk)}")

# Visualize: decision regions on Iris (first 2 features)
iris = load_iris()
X2, y = iris.data[:, :2], iris.target
X2_tr, X2_te, y_tr, y_te = train_test_split(X2, y, test_size=0.3, random_state=42)
gnb2 = GaussianNaiveBayes().fit(X2_tr, y_tr)

xx, yy = np.meshgrid(np.linspace(X2[:,0].min()-0.5, X2[:,0].max()+0.5, 200),
                      np.linspace(X2[:,1].min()-0.5, X2[:,1].max()+0.5, 200))
Z = gnb2.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
colors_cls = ['#e74c3c','#2ecc71','#3498db']
ax1.contourf(xx, yy, Z, alpha=0.2, cmap='RdYlBu')
for c, col in enumerate(colors_cls):
    mask = y == c
    ax1.scatter(X2[mask,0], X2[mask,1], c=col, s=30, alpha=0.7, label=iris.target_names[c])
ax1.set_xlabel(iris.feature_names[0]); ax1.set_ylabel(iris.feature_names[1])
ax1.set_title('Gaussian Naive Bayes Decision Regions\n(Iris, first 2 features)', fontsize=11)
ax1.legend()

# Right: posterior probabilities bar chart for test samples
probs = gnb2.predict_proba(X2_te)
sample_indices = [0, 1, 5, 10, 15]
sample_probs = probs[sample_indices]
x_pos = np.arange(len(sample_indices))
for cls_idx, col in enumerate(colors_cls):
    ax2.bar(x_pos + cls_idx*0.25, sample_probs[:, cls_idx], 0.25,
            color=col, alpha=0.75, label=iris.target_names[cls_idx])
ax2.set_xticks(x_pos + 0.25)
ax2.set_xticklabels([f'Test[{i}]\ny={y_te[i]}' for i in sample_indices])
ax2.set_ylabel('Posterior probability P(Y=c|x)')
ax2.set_title('Predicted Class Probabilities\n(5 test samples)', fontsize=11)
ax2.legend(); ax2.grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.show()

In [ ]:
# ─── CELL 2: Multinomial Naive Bayes — Text Classification with Laplace Smoothing ──
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

class MultinomialNaiveBayes:
    """
    Multinomial Naive Bayes for text classification.
    Input X: word count matrix (N x V), y: class labels.
    """
    def __init__(self, alpha=1.0):
        self.alpha = alpha  # Laplace smoothing (Dirichlet prior pseudo-count)

    def fit(self, X, y):
        N, V = X.shape
        self.classes_ = np.unique(y)

        # Class priors = log P(Y=c)
        self.log_priors_ = np.array([np.log(np.mean(y == c)) for c in self.classes_])

        # Word likelihoods: theta[c, w] = P(word w | class c)
        # MAP with Dirichlet(alpha) prior = Laplace smoothing
        self.log_theta_ = np.zeros((len(self.classes_), V))
        for i, c in enumerate(self.classes_):
            Xc = X[y == c]  # documents of class c
            word_counts = Xc.sum(axis=0) + self.alpha          # add pseudo-count
            total_words = word_counts.sum()                     # total + alpha*V
            self.log_theta_[i] = np.log(word_counts / total_words)  # log P(w|c)
        return self

    def predict_log_proba(self, X):
        # log P(Y=c|X) ∝ log P(Y=c) + sum_w n_w * log theta_{wc}
        return X @ self.log_theta_.T + self.log_priors_  # (N, C)

    def predict(self, X):
        return self.classes_[self.predict_log_proba(X).argmax(axis=1)]


# ── Synthetic Spam Dataset ────────────────────────────────────────────────────
np.random.seed(42)
vocab = ['free', 'money', 'click', 'win', 'prize', 'lottery',  # spam words
         'meeting', 'project', 'report', 'schedule', 'team', 'update']  # ham words
V = len(vocab)

def gen_doc(n_words, word_probs):
    words = np.random.choice(V, size=n_words, p=word_probs)
    counts = np.bincount(words, minlength=V)
    return counts

spam_probs = np.array([0.15, 0.12, 0.11, 0.10, 0.10, 0.10, 0.06, 0.06, 0.05, 0.05, 0.05, 0.05])
ham_probs  = np.array([0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.15, 0.15, 0.13, 0.13, 0.13, 0.13])

N_spam, N_ham = 200, 300
X_spam = np.array([gen_doc(np.random.randint(50, 150), spam_probs) for _ in range(N_spam)])
X_ham  = np.array([gen_doc(np.random.randint(50, 150), ham_probs)  for _ in range(N_ham)])
X = np.vstack([X_spam, X_ham])
y = np.array([0]*N_spam + [1]*N_ham)  # 0=spam, 1=ham

# Train/test split
perm = np.random.permutation(len(y))
X, y = X[perm], y[perm]
split = int(0.8 * len(y))
X_tr, X_te, y_tr, y_te = X[:split], X[split:], y[:split], y[split:]

# Fit with different smoothing values
alphas = [0.0001, 0.01, 0.1, 1.0, 5.0, 20.0]
accs = []
for alpha in alphas:
    mnb = MultinomialNaiveBayes(alpha=max(alpha, 1e-9)).fit(X_tr, y_tr)
    acc = (mnb.predict(X_te) == y_te).mean()
    accs.append(acc)
    print(f"α={alpha:6.4f}: test accuracy = {acc:.4f}")

# Fit final model with alpha=1
mnb = MultinomialNaiveBayes(alpha=1.0).fit(X_tr, y_tr)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Left: word probabilities per class
x_idx = np.arange(V)
theta_spam = np.exp(mnb.log_theta_[0])
theta_ham  = np.exp(mnb.log_theta_[1])
axes[0].barh(x_idx - 0.2, theta_spam, 0.35, color='red', alpha=0.7, label='Spam P(w|spam)')
axes[0].barh(x_idx + 0.2, theta_ham, 0.35, color='steelblue', alpha=0.7, label='Ham P(w|ham)')
axes[0].set_yticks(x_idx); axes[0].set_yticklabels(vocab)
axes[0].set_xlabel('Word probability θ_{wc}')
axes[0].set_title('Learned Word Probabilities per Class', fontsize=11)
axes[0].legend(); axes[0].grid(alpha=0.3, axis='x')

# Middle: effect of smoothing on accuracy
axes[1].semilogx(alphas, accs, 'o-', color='darkorange', lw=2.5, ms=8)
axes[1].set_xlabel('Laplace smoothing α (Dirichlet prior strength)')
axes[1].set_ylabel('Test accuracy')
axes[1].set_title('Effect of Laplace Smoothing (α) on Test Accuracy\nα=1 is the standard choice', fontsize=10)
axes[1].grid(alpha=0.3)

# Right: confusion matrix
y_pred = mnb.predict(X_te)
cm = np.array([[((y_te==0) & (y_pred==0)).sum(), ((y_te==0) & (y_pred==1)).sum()],
               [((y_te==1) & (y_pred==0)).sum(), ((y_te==1) & (y_pred==1)).sum()]])
im = axes[2].imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        axes[2].text(j, i, str(cm[i,j]), ha='center', va='center', fontsize=18, fontweight='bold',
                     color='white' if cm[i,j] > cm.max()*0.6 else 'black')
axes[2].set_xticks([0,1]); axes[2].set_xticklabels(['Pred: Spam','Pred: Ham'])
axes[2].set_yticks([0,1]); axes[2].set_yticklabels(['True: Spam','True: Ham'])
axes[2].set_title(f'Confusion Matrix (α=1.0)\nAccuracy={cm.diagonal().sum()/cm.sum():.3f}', fontsize=11)

plt.suptitle('Multinomial Naive Bayes for Spam Detection', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()